# Financial Fraud Detection (Pix)

Este projeto simula um motor de detecção de fraudes para transações financeiras móveis (semelhante ao Pix). O objetivo é processar um grande volume de logs transacionais para identificar padrões anômalos e blindar o sistema contra perdas financeiras.

Diferente de datasets didáticos pequenos, este projeto utiliza o **PaySim**, contendo mais de **6 milhões de registros**, exigindo o uso de tecnologias de Big Data (Spark) para processamento distribuído, já que ferramentas tradicionais (Excel/Pandas local) não performam adequadamente nesta escala.

- **Desafio de Negócio:** Detectar a minoria fraudulenta (0.1% dos casos) sem bloquear clientes legítimos (Falsos Positivos), lidando com o severo desbalanceamento de classes.
## 🛠 Tecnologias Utilizadas

* **Linguagem:** Python (PySpark API)
* **Processamento:** Apache Spark (Computação Distribuída & In-Memory Processing)
* **Armazenamento:** Databricks File System (DBFS) e Delta Lake (Camadas Bronze/Silver)
* **Machine Learning:** XGBoost, Scikit-learn, MLflow
* **Ambiente:** Databricks Community Edition (Serverless Compute)
## 📂 Dados

O dataset utilizado é o **PaySim: Mobile Money Simulator**, criado a partir de logs reais de transações financeiras anonimizadas.

* **Fonte:** [Kaggle - PaySim Dataset](https://www.kaggle.com/datasets/ealaxi/paysim1)
* **Volume:** ~6.3 milhões de transações (Simulação de Big Data Real)
* **Tamanho:** ~470MB (CSV Bruto)
* **Autor:** Edgar Lopez-Rojas



## Imports, Schema, Bronze -> Silver... Passos Iniciais

In [0]:
# Imports

from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import pandas as pd
sns.set_style("dark")
plt.style.use('dark_background')

In [0]:
# ============================================
# Plot Config
# ============================================

BACKGROUND   = "#0D1117"   # GitHub dark background
SURFACE      = "#161B22"   # Card/panel surface
GRID         = "#21262D"   # Grid lines
TEXT_PRIMARY = "#E6EDF3"   # Títulos e labels
TEXT_MUTED   = "#7D8590"   # Subtítulos e ticks
ACCENT_BLUE  = "#58A6FF"   # Destaque principal (legítimo)
ACCENT_RED   = "#FF4D4D"   # Destaque alerta (fraude)
ACCENT_GREEN = "#3FB950"   # Confirmação/insight
ACCENT_AMBER = "#D29922"   # Atenção/neutro

plt.rcParams.update({
    # Fundo
    "figure.facecolor":  BACKGROUND,
    "axes.facecolor":    SURFACE,
    "savefig.facecolor": BACKGROUND,

    # Grid
    "axes.grid":          True,
    "grid.color":         GRID,
    "grid.linewidth":     0.6,
    "grid.linestyle":     "--",

    # Bordas dos eixos
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.spines.left":   False,
    "axes.spines.bottom": False,

    # Texto
    "text.color":         TEXT_PRIMARY,
    "axes.labelcolor":    TEXT_PRIMARY,
    "xtick.color":        TEXT_MUTED,
    "ytick.color":        TEXT_MUTED,
    "axes.titlecolor":    TEXT_PRIMARY,
    "axes.titlesize":     14,
    "axes.titleweight":   "bold",
    "axes.labelsize":     11,
    "xtick.labelsize":    10,
    "ytick.labelsize":    10,

    # Fonte
    "font.family":        "monospace",  # vibe tech/code

    # Figura
    "figure.dpi":         150,
    "figure.figsize":     (10, 5),

    # Legenda
    "legend.facecolor":   SURFACE,
    "legend.edgecolor":   GRID,
    "legend.labelcolor":  TEXT_PRIMARY,
})

# Paleta padrão seaborn
sns.set_palette([ACCENT_BLUE, ACCENT_RED, ACCENT_GREEN, ACCENT_AMBER])

# Função helper para salvar
def save_plot(filename):
    plt.tight_layout()
    plt.savefig(f"imgs/{filename}", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

In [0]:
# Criando a base de dados "fraud_det_project"

spark.sql("USE CATALOG workspace") 
spark.sql("CREATE SCHEMA IF NOT EXISTS fraud_det_project")
spark.sql("USE fraud_det_project")
print("Banco de dados 'fraud_det_project' criado e selecionado!")

In [0]:
# Lendo os dados e criando a tabela Bronze

df_bronze = spark.read.table("workspace.default.paysim_bronze")

print("Tabela encontrada! Contagem de linhas:")
print(f"\nTotal de linhas no dataset: {df_bronze.count()}")


`step` = Unidade de tempo. Hora em que aconteceu a transação. Vai de 1 a 743.

- Vamos particionar pelo `step`

In [0]:
%sql

-- Casting usando SQL e criando a tabela Silver --

CREATE OR REPLACE TABLE paysim_silver
USING DELTA
PARTITIONED BY (step)
AS
SELECT 
    CAST(step AS INT) as step,
    CAST(type AS STRING) as type,
    CAST(amount AS DOUBLE) as amount,
    CAST(nameOrig AS STRING) as name_orig,
    CAST(oldbalanceOrg AS DOUBLE) as old_balance_org,
    CAST(newbalanceOrig AS DOUBLE) as new_balance_orig,
    CAST(nameDest AS STRING) as name_dest,
    CAST(oldbalanceDest AS DOUBLE) as old_balance_dest,
    CAST(newbalanceDest AS DOUBLE) as new_balance_dest,
    CAST(isFraud AS INT) as is_fraud,
    CAST(isFlaggedFraud AS INT) as is_flagged_fraud
    
FROM workspace.default.paysim_bronze;

In [0]:
# Lendo o a tabela Silver

df_silver = spark.read.table("paysim_silver")

### Removendo Duplicatas e Tratando Nulos

In [0]:
# Remove duplicatas
df_clean = df_silver.dropDuplicates()

# Contagem de nulos por coluna
null_counts = df_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_clean.columns
])

display(null_counts)

## FEATURES - ERRO Matemático da Transação e Normalizando `step`

| Feature | O que é? |
| --- | --- |
| **`error_orig`**      | A diferença matemática no saldo da vítima.  |
| **`error_dest`**      | A diferença matemática no saldo do destino. |
| **`hour_of_day`**     | Conversão de `step` para hora (0-23h).      |

In [0]:
# Tipos onde a conta de origem RECEBE dinheiro (sinal invertido no erro)
incoming_types = ["CASH_IN"]

# Erro contábil da origem: diferença entre saldo esperado e saldo real após transação
df_with_error_orig = df_clean.withColumn("error_orig",
    F.when(F.col("type").isin(incoming_types),
         F.col("old_balance_org") + F.col("amount") - F.col("new_balance_orig")
    ).otherwise(
         F.col("old_balance_org") - F.col("amount") - F.col("new_balance_orig")
    )
)

# Erro contábil do destino: merchants (prefixo "M") não têm dados de saldo
df_with_errors = df_with_error_orig.withColumn("error_dest",
    F.when(F.col("name_dest").startswith("M"),
         F.lit(0.0)
    ).when(F.col("type").isin(incoming_types),
         F.col("old_balance_dest") - F.col("amount") - F.col("new_balance_dest")
    ).otherwise(
         F.col("old_balance_dest") + F.col("amount") - F.col("new_balance_dest")
    )
)

# Amostra de validação no step 496

display(df_with_errors.select("type", "amount", "is_fraud", "error_orig", "error_dest")
                      .filter(F.col("step") == 496)
                      .limit(5))

In [0]:
# Converte step em hora do dia (ciclo de 24h)

df_with_errors = df_with_errors.withColumn("hour_of_day", F.col("step") % 24)

### Interpretando em `error_orig/dest`
| Variável | Sinal | Significado | Grau de Suspeita |
| :--- | :---: | :--- | :--- |
| **error_orig** | **Negativo (-)** | **Dinheiro preso:** O valor deveria ter saído da conta de origem, mas o saldo **não baixou** (ou baixou menos do que devia). |  **FRAUDE CRÍTICA** *(O fraudador transfere/saca sem que o débito ocorra na conta).* |
| **error_orig** | **Positivo (+)** | **Desconto excessivo:** O saldo da origem baixou **mais** do que o valor da transação. |  **Baixa** (Geralmente indica erro de sistema ou cobrança de taxas ocultas). |
| **error_dest** | **Positivo (+)** | **Dinheiro sumiu:** O valor deveria ter entrado na conta destino, mas o saldo **não subiu**. |  **LAVAGEM / CONTA LARANJA** *(Indica conta "fantasma" ou saque imediato para não deixar rastro).* |
| **error_dest** | **Negativo (-)** | **Dinheiro brotou:** O saldo do destino subiu **mais** do que o valor enviado. |  **Baixa** *(Geralmente erro de sincronização do sistema).* |

> Para considerar fraude, o erro deve ser significativo. Erros muito pequenos (ex: `1.16e-10`) devem ser ignorados usando um *threshold* de tolerância (ex: `abs(error) > 0.05`).

## EXPLORATORY DATA ANALYSIS (EDA)

In [0]:
# Total de transações por tipo
display(df_with_errors.groupBy("type")
                      .count()
                      .orderBy("count", ascending=False))

In [0]:
# Média e mediana do valor transacionado por tipo
df_amount_stats = df_with_errors.groupBy("type").agg(
    F.avg("amount").alias("mean_amount"),
    F.percentile_approx("amount", 0.5).alias("median_amount")
).orderBy("mean_amount", ascending=False)

display(df_amount_stats)

In [0]:
# Top 5 contas de origem com mais fraudes — padrão Hit & Run (uma fraude por conta)
df_with_errors.filter(F.col("is_fraud") == 1) \
    .groupBy("name_orig") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(5)

# Top 5 contas de destino com mais fraudes — padrão mula descartável (reuso mínimo)
df_with_errors.filter(F.col("is_fraud") == 1) \
    .groupBy("name_dest") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(5)

## Provas e Falhas de Fraude por Lavagem

> **A Tese do Esvaziamento (Account Takeover)**: Indiferente da complexidade, a fraude financeira tende a seguir a regra universal do **Esvaziamento**. Em um cenário de _Account Takeover_ (Tomada de Conta), o criminoso assume o controle, transfere o saldo total para uma conta "laranja" (pivô) e realiza o saque imediatamente. O objetivo é ocultar a origem (Lavagem de Dinheiro) e não deixar rastro financeiro.

### Prova 1: Fluxo de Entrada

> Hipótese: A fraude só existe em dois momentos: na saída do dinheiro da vítima e no saque do criminoso. Roubo e saque.

In [0]:
# Contagem de fraudes por tipo de transação
df_fraud_by_type = df_with_errors.groupBy("type") \
    .agg(F.sum("is_fraud").alias("total_fraud")) \
    .orderBy("total_fraud", ascending=False)

display(df_fraud_by_type)

In [0]:
# Prova 1: Acoplamento 1:1 — fraudes concentradas em TRANSFER e CASH_OUT
flow_proof_pdf = df_fraud_by_type.toPandas()

fig, ax = plt.subplots()

sns.barplot(
    data=flow_proof_pdf,
    x="type",
    y="total_fraud",
    color=ACCENT_RED,
    ax=ax,
    edgecolor="none"
)

ax.set_ylim(0, flow_proof_pdf["total_fraud"].max() * 1.15)
ax.bar_label(ax.containers[0], padding=8, fontweight="bold", fontsize=12, color=TEXT_PRIMARY)

ax.set_title("Prova de Fluxo: Acoplamento 1:1", pad=20)
ax.set_xlabel("Tipo de Transação", labelpad=15)
ax.set_ylabel("Total de Fraudes", labelpad=15)
ax.tick_params(axis="x", pad=10)
plt.xticks(rotation=45, ha="right")

save_plot("flow_proof.png")


1. Existe um acoplamento quase perfeito **(1:1)**. Para cada Transferência (Roubo da Vítima), existe um Saque (Mula retirando o dinheiro). 
2. Tipos como `PAYMENT` (pagar boleto) ou `CASH_IN` (depósito) são ruído. Deve-se focar somente em `TRANSFER` e `CASH_OUT`.


In [0]:
# Filtra somente TRANSFER e CASH_OUT — tipos relevantes para detecção de fraude
df_fraud_focus = df_with_errors.filter(F.col("type").isin(["TRANSFER", "CASH_OUT"]))

print(f"Registros no escopo: {df_fraud_focus.count():,}")

### Prova 2: Padrão de Esvaziamento

> Hipótese: Se fizermos um gráfico nas instância onde há fraude confirmada do tipo transferência, o fraudador tenta levar o máximo possível


In [0]:
# Prova 2: Esvaziamento — valor transferido igual ao saldo da vítima (y = x)
emptying_proof_pdf = df_fraud_focus \
    .filter((F.col("is_fraud") == 1) & (F.col("type") == "TRANSFER")) \
    .select("old_balance_org", "amount") \
    .toPandas()

fig, ax = plt.subplots()

sns.scatterplot(
    data=emptying_proof_pdf,
    x="old_balance_org",
    y="amount",
    alpha=0.6,
    color=ACCENT_RED,
    edgecolor=None,
    ax=ax
)

# Linha de referência: esvaziamento total (y = x)
max_val = max(emptying_proof_pdf["old_balance_org"].max(), emptying_proof_pdf["amount"].max())
ax.plot([0, max_val], [0, max_val],
    color=TEXT_MUTED,
    linestyle="--",
    linewidth=1.5,
    label="Esvaziamento Total (Transferência = Saldo)"
)

ax.set_title("A Regra do Esvaziamento: Transferência de 100% do Saldo", pad=20)
ax.set_xlabel("Saldo Original da Vítima", labelpad=15)
ax.set_ylabel("Valor Transferido", labelpad=15)
ax.tick_params(axis="both", pad=10)
ax.legend(frameon=False, loc="upper left")

save_plot("emptying_proof.png")


1. A linha perfeita de 45 graus confirma que, na esmagadora maioria dos casos, `Valor = Saldo`.
2. Além disso, notamos um teto horizontal em **10 Milhões**. Isso revela que, mesmo que a vítima tenha 60 milhões, o fraudador bate no limite transacional do sistema.

### Prova 3: O rastro das Contas Laranja (Mulas)

> Hipótese: Se é um roubo seguido de saque, a conta de destino deve começar zerada. Contas receptoras (mulas) são descartáveis e nascem zeradas para receber o ilícito.

In [0]:
# Prova 3: Padrão mula — contas de destino descartáveis, nascidas zeradas
mule_proof_pdf = df_fraud_focus \
    .filter((F.col("is_fraud") == 1) & (F.col("type") == "TRANSFER")) \
    .withColumn(
        "mule_type",
        F.when((F.col("old_balance_dest") == 0) & (F.col("new_balance_dest") == F.col("amount")), "Perfect Mule (Stays)")
         .when((F.col("old_balance_dest") == 0) & (F.col("new_balance_dest") == 0), "Balance Failure (Vanishes)")
         .otherwise("Account with Prior Balance")
    ) \
    .groupBy("mule_type") \
    .count() \
    .toPandas()

In [0]:
# Prova 3: Distribuição dos padrões de conta mula
total_frauds = mule_proof_pdf["count"].sum()
mule_proof_pdf["percentage"] = (mule_proof_pdf["count"] / total_frauds) * 100
mule_proof_pdf = mule_proof_pdf.sort_values("percentage", ascending=False)

fig, ax = plt.subplots()

bar_colors = [ACCENT_RED if p > 50 else TEXT_MUTED for p in mule_proof_pdf["percentage"]]

sns.barplot(
    data=mule_proof_pdf,
    x="percentage",
    y="mule_type",
    hue="mule_type",
    palette=bar_colors,
    legend=False,
    ax=ax,
    edgecolor="none"
)

for i, (pct, count) in enumerate(zip(mule_proof_pdf["percentage"], mule_proof_pdf["count"])):
    ax.text(pct + 1, i, f"{pct:.1f}% ({count:,})", va="center", fontweight="bold", color=TEXT_PRIMARY)

ax.set_title("Prova das Contas Mulas", pad=20)
ax.set_ylabel("")
ax.set_xlim(0, 115)
ax.get_xaxis().set_visible(False)
ax.tick_params(axis="y", length=0, pad=10)
sns.despine(left=True, bottom=True)

save_plot("mules_proof.png")


- **Laranja com "Erro" de Saldo (99.3%):** A conta estava zerada, recebeu o dinheiro, mas o log de `newBalanceDest` permaneceu zerado ou inconsistente.
- **Conta com Saldo Prévio (0.7%):** Contas que já tinham dinheiro (mistura de fundos).
- **Laranja Perfeito (0%):** Onde a matemática bate exata.

### Falha 1: A Rede de Lavagem

> Hipótese: A conta que recebe a transferência fraudulenta (name_dest em TRANSFER) é a mesma conta que realiza o saque logo em seguida (name_orig em CASH_OUT).

Resposta: Falha. O PaySim não mantém a consistência da identidade da mula ao longo do tempo (IDs reutilizados ou aleatórios).

- Há somente uma instância onde isso ocorre no conjunto inteiro.

In [0]:
# Hipótese: conta que recebe TRANSFER fraudulento é a mesma que faz CASH_OUT (rede de lavagem)
df_fraud_transfers = df_fraud_focus \
    .filter((F.col("is_fraud") == 1) & (F.col("type") == "TRANSFER")) \
    .select(
        F.col("step").alias("step_in"),
        F.col("name_dest").alias("mule_account"),
        F.col("amount").alias("amount_in")
    )

df_cashouts = df_fraud_focus \
    .filter(F.col("type") == "CASH_OUT") \
    .select(
        F.col("step").alias("step_out"),
        F.col("name_orig").alias("mule_account_out"),
        F.col("amount").alias("amount_out")
    )

# Join tentando rastrear o ciclo completo: roubo → lavagem
df_laundering_cycle = df_fraud_transfers.join(
    df_cashouts,
    on=df_fraud_transfers.mule_account == df_cashouts.mule_account_out,
    how="inner"
)

# Tempo entre o roubo e o saque (em steps/horas)
df_laundering_timing = df_laundering_cycle \
    .filter(F.col("step_out") >= F.col("step_in")) \
    .withColumn("steps_to_launder", F.col("step_out") - F.col("step_in")) \
    .orderBy("steps_to_launder")

display(df_laundering_timing)


### Falha 2: O Horário do Crime

> Hipótese: Fraudes geralmente acontecem de madrugada.

Resposta: Falha. Volume absoluto de fraudes é constante nas 24h.

- Por demonstrar intensidade similar durante o dia inteiro, pode ser prova do uso de bots.

In [0]:
# Total de fraudes por hora do dia — teste da hipótese de pico noturno
df_fraud_by_hour = df_with_errors.groupBy("hour_of_day") \
    .agg(F.sum("is_fraud").alias("total_fraud")) \
    .orderBy("total_fraud", ascending=False)

display(df_fraud_by_hour.limit(5))

## FEATURE  - `ratio_amount_balance`

| Feature | O que é? |
| --- | --- |
| **`ratio_amount_balance`**    | Relação Valor / Saldo.              |

- 1.0 = Esvaziamento total.
- \>1.0 = Cheque especial/Bug (levou mais do que tinha) - Geralmente legítima.
- -1.0 = O saldo já era zero (erro/anomalia.)

In [0]:
# Feature: razão entre valor transferido e saldo disponível (1.0 = esvaziamento total)
df_fraud_focus = df_fraud_focus.withColumn("ratio_amount_balance",
    F.when(F.col("old_balance_org") > 0,
           F.col("amount") / F.col("old_balance_org")
    ).otherwise(F.lit(-1.0))  # -1.0 para saldo zero (evita divisão por zero)
)

# Validação: casos de esvaziamento total confirmados (ratio = 1.0)
display(df_fraud_focus.filter(F.col("ratio_amount_balance") == 1.0)
                      .select("type", "amount", "old_balance_org", "ratio_amount_balance", "is_fraud")
                      .limit(5))

%md
## FEATURES  - Ratio Amount

| Feature | O que é? |
| --- | --- |
| **`dest_is_empty`**            | Classificação textual ("Laranja Perfeito"). |
| **`type_idx`**                 | "TRANSFER" ou "CASH_OUT".                     |

In [0]:
# Encoding de variáveis categóricas e flags para o modelo
df_transformed = df_fraud_focus \
    .withColumn("type_idx",
                F.when(F.col("type") == "TRANSFER", F.lit(0))  # 0 = TRANSFER
                 .otherwise(F.lit(1))                           # 1 = CASH_OUT
    ) \
    .withColumn("dest_is_empty",
                F.when(F.col("old_balance_dest") == 0, F.lit(1))  # 1 = mula descartável
                 .otherwise(F.lit(0))                              # 0 = conta com histórico
    )

## Consolidando o DataFrame

In [0]:
# Seleção das features para a Analytical Base Table (ABT)
feature_columns = [
    "step",
    "hour_of_day",
    "type_idx",
    "amount",
    "old_balance_org",
    "error_orig",
    "old_balance_dest",
    "error_dest",
    "dest_is_empty",
    "ratio_amount_balance",
    "is_fraud"
]

df_abt = df_transformed.select(feature_columns)

In [0]:
# Persiste a ABT na camada Gold como Delta Table
df_abt.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("paysim_gold")